# Analisis Distribution Shift: Sintetik (CIC, UNS) vs Real-Traffic (AWS)

**Dijalankan di SageMaker.** Notebook ini membandingkan karakteristik distribusi **9 fitur SFM**
antara dataset sintetik (CSE-CIC-IDS2018 = **CIC**, UNSW-NB15 = **UNS**) dan trafik **nyata AWS**
(hasil capture Fase 1/2). Membuktikan *distribution shift* secara kuantitatif & visual
(pendekatan Wasserstein a la Layeghy dkk. + ECDF/boxplot/PCA/t-SNE/domain-classifier).

**Alur artefak (penting):**
1. SageMaker: muat fitur CIC/UNS (data latih) + CSV fitur AWS (dari S3 `unsw-far/results/`).
2. Hitung semua metrik -> kumpulkan ke `RESULTS` (dict) + simpan grafik PNG.
3. Simpan `dataset_shift_results.json` + PNG, lalu **UPLOAD ke S3** `unsw-far/shift/`.
4. (Di PC lokal) unduh artefak dari S3 untuk analisis/diskusi lanjutan.

> Sediakan CSV 9-fitur CIC & UNS (`cic_flows.csv`/`uns_flows.csv`) atau biarkan notebook
> mengekstraknya dari sumber latih (lihat Sel 1B). Jika CIC/UNS tak tersedia, dibuat sampel
> sintetik dari scaler (ILUSTRATIF, ditandai `SYNTHETIC=True`).

## 0. Setup & konfigurasi

In [ ]:
import importlib, sys, subprocess
pkgmap = {'sklearn':'scikit-learn'}
need = [m for m in ('matplotlib','pandas','numpy','scipy','sklearn','boto3') if importlib.util.find_spec(m) is None]
if need:
    subprocess.run([sys.executable,'-m','pip','install','-q',*[pkgmap.get(m,m) for m in need]], check=True)
print('setup ok' if not need else f'installed: {need}')

In [ ]:
import os, json, glob, datetime
import numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')  # simpan gambar tanpa display; ganti ke inline bila ingin lihat
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.grid': True, 'grid.alpha': 0.3})

# ---- KONFIGURASI (sesuaikan bila perlu) ----
S3_BUCKET = os.environ.get('S3_BUCKET', 'ssh-detection-features-232032302717')
S3_PREFIX = 'unsw-far'
REGION    = os.environ.get('AWS_REGION', 'ap-southeast-1')
OUTDIR    = 'shift_out'          # folder lokal artefak
os.makedirs(OUTDIR, exist_ok=True)

CANON = ['duration','fwd_pkts','bwd_pkts','fwd_bytes','bwd_bytes','fwd_mean','bwd_mean','src_load','dst_load']
colors = {'CIC':'#4C72B0','UNS':'#DD8452','AWS-real':'#55A868'}
RESULTS = {'generated': datetime.datetime.utcnow().isoformat()+'Z', 'features': CANON}
def savefig(name):
    p = os.path.join(OUTDIR, name); plt.savefig(p, bbox_inches='tight'); plt.close(); print('  saved', p); return p

## 1A. Muat fitur AWS (dari S3 `unsw-far/results/` atau lokal)
Unduh CSV `*_flows.csv` dari S3 bila belum ada lokal, lalu gabung.

In [ ]:
# Unduh CSV AWS dari S3 -> folder aws_flows/
AWS_DIR = 'aws_flows'
os.makedirs(AWS_DIR, exist_ok=True)
try:
    import boto3
    s3 = boto3.client('s3', region_name=REGION)
    pref = f'{S3_PREFIX}/results/'
    objs = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=pref).get('Contents', [])
    got = 0
    for o in objs:
        key = o['Key']
        if key.endswith('_flows.csv'):
            dst = os.path.join(AWS_DIR, os.path.basename(key))
            if not os.path.exists(dst):
                s3.download_file(S3_BUCKET, key, dst)
            got += 1
    print(f'Unduh {got} CSV AWS dari s3://{S3_BUCKET}/{pref}')
except Exception as e:
    print('S3 download dilewati (mungkin sudah ada lokal / tak ada akses):', e)

def read_feats(files):
    dfs=[]
    for f in files:
        try:
            d = pd.read_csv(f)
            if all(c in d.columns for c in CANON): dfs.append(d[CANON])
        except Exception as ex: print('  skip',f,ex)
    return pd.concat(dfs, ignore_index=True) if dfs else None

aws_files = sorted(glob.glob(os.path.join(AWS_DIR,'*_flows.csv'))) or sorted(glob.glob('/opt/unsw/results/*_flows.csv'))
aws = read_feats(aws_files)
print('AWS flows:', 0 if aws is None else len(aws), '| files:', [os.path.basename(f) for f in aws_files])

## 1B. Muat fitur CIC & UNS (data latih)

Prioritas: CSV 9-fitur siap (`cic_flows.csv`,`uns_flows.csv`). Jika tidak ada, notebook mencoba
membangunnya dari dataset latih SFM yang biasa dipakai (sesuaikan `CIC_SRC`/`UNS_SRC` ke path
dataset kamu di SageMaker). Jika tetap gagal -> sampel sintetik (ILUSTRATIF).

In [ ]:
# --- Opsi 1: CSV 9-fitur siap pakai (bila sudah pernah diekspor) ---
def first(paths):
    for p in paths:
        h = sorted(glob.glob(p))
        if h: return h
    return []
cic = read_feats(first(['cic_flows.csv','*cic*flows*.csv','../cic_flows.csv']))
uns = read_feats(first(['uns_flows.csv','*uns*flows*.csv','../uns_flows.csv']))

# --- Opsi 2: bangun 9 fitur CANON dari dataset MENTAH (CIC pkl + UNSW csv) ---
# Pemetaan kolom & satuan mengikuti 01_feature_inventory & 02_mapping_validation.
# CANON:  duration fwd_pkts bwd_pkts fwd_bytes bwd_bytes fwd_mean bwd_mean src_load dst_load
# CIC  :  Flow Duration(us), Tot Fwd Pkts, Tot Bwd Pkts, TotLen Fwd Pkts, TotLen Bwd Pkts,
#         Fwd Pkt Len Mean, Bwd Pkt Len Mean, Flow Byts/s, Bwd Pkts/s
# UNSW :  dur(detik->x1e6=us), spkts, dpkts, sbytes, dbytes, smean, dmean, sload, dload
CIC_MAP = {'duration':'Flow Duration','fwd_pkts':'Tot Fwd Pkts','bwd_pkts':'Tot Bwd Pkts',
           'fwd_bytes':'TotLen Fwd Pkts','bwd_bytes':'TotLen Bwd Pkts','fwd_mean':'Fwd Pkt Len Mean',
           'bwd_mean':'Bwd Pkt Len Mean','src_load':'Flow Byts/s','dst_load':'Bwd Pkts/s'}
UNS_MAP = {'duration':'dur','fwd_pkts':'spkts','bwd_pkts':'dpkts','fwd_bytes':'sbytes',
           'bwd_bytes':'dbytes','fwd_mean':'smean','bwd_mean':'dmean','src_load':'sload','dst_load':'dload'}
CIC_PKL = first(['../../CICDDoS2018/data/cleaned_100.pkl','../../CICDDoS2018/data/cleaned_*.pkl'])
UNS_CSV = first(['../data/UNSW_NB15_testing-set.csv','../data/UNSW_NB15_training-set.csv','../data/UNSW_NB15_*set.csv'])

if cic is None and CIC_PKL:
    try:
        import pickle
        with open(CIC_PKL[0],'rb') as f: d=pickle.load(f)
        feats=list(d['feature_names']); X=np.asarray(d['X'],float)
        sc=d.get('scaler',None)
        Xo = X*sc.scale_+sc.mean_ if (sc is not None and hasattr(sc,'scale_')) else X
        cdf=pd.DataFrame(Xo,columns=feats)
        miss=[v for v in CIC_MAP.values() if v not in cdf.columns]
        if miss: print('  CIC kolom hilang:',miss)
        else:
            cic=pd.DataFrame({k:cdf[CIC_MAP[k]].values for k in CANON})[CANON].astype(float)
            print('CIC dibangun dari',os.path.basename(CIC_PKL[0]),'->',len(cic),'baris (satuan asli)')
    except Exception as e: print('  gagal bangun CIC:',e)

if uns is None and UNS_CSV:
    try:
        udf=pd.read_csv(UNS_CSV[0])
        miss=[v for v in UNS_MAP.values() if v not in udf.columns]
        if miss: print('  UNS kolom hilang:',miss)
        else:
            uns=pd.DataFrame({k:pd.to_numeric(udf[UNS_MAP[k]],errors='coerce').values for k in CANON})[CANON].astype(float)
            uns['duration']=uns['duration']*1e6  # dur detik -> mikrodetik (samakan CIC/AWS)
            uns=uns.replace([np.inf,-np.inf],np.nan).dropna()
            print('UNS dibangun dari',os.path.basename(UNS_CSV[0]),'->',len(uns),'baris (dur x1e6=us)')
    except Exception as e: print('  gagal bangun UNS:',e)

# (opsional) ekspor ke CSV agar run berikutnya lebih cepat
if cic is not None and not os.path.exists('cic_flows.csv'):
    try: cic.to_csv('cic_flows.csv',index=False); print('  ekspor cic_flows.csv')
    except Exception: pass
if uns is not None and not os.path.exists('uns_flows.csv'):
    try: uns.to_csv('uns_flows.csv',index=False); print('  ekspor uns_flows.csv')
    except Exception: pass

SYNTHETIC = (cic is None) or (uns is None)
if SYNTHETIC:
    # sampel sintetik dari scaler deployment (ilustrasi bentuk saja)
    dm = None
    for p in ['deploy_meta_9feat.json','../deploy_meta_9feat.json','models/deploy_meta_9feat.json']:
        if os.path.exists(p): dm = json.load(open(p)); break
    if dm is None:
        mean = np.array([1.2175e7,24.43,6.21,990.47,4609.18,50.32,113.18,255117.10,15260.12])
        scale= np.array([2.0e7,60,20,5000,20000,60,150,6.0e5,4.0e4])
    else:
        mean=np.array(dm['scaler_mean']); scale=np.array(dm['scaler_scale'])
    rng=np.random.default_rng(42); n = len(aws) if aws is not None else 3000
    cic = pd.DataFrame(np.abs(rng.normal(mean,np.abs(scale),size=(n,9))),columns=CANON)
    uns = pd.DataFrame(np.abs(rng.normal(mean*np.array([1e-6,1,2,0.4,0.2,1.4,0.5,3.5,0.1]),np.abs(scale)*0.8,size=(n,9))),columns=CANON)
    print('CIC/UNS CSV tidak ada -> SAMPEL SINTETIK (ilustratif).')
else:
    print('CIC flows:', len(cic), '| UNS flows:', len(uns))

domains = {k:v for k,v in [('CIC',cic),('UNS',uns),('AWS-real',aws)] if v is not None and len(v)>0}
RESULTS['synthetic_cic_uns'] = bool(SYNTHETIC)
RESULTS['n_flow'] = {k:int(len(v)) for k,v in domains.items()}
print('Domain:', list(domains.keys()), '| SYNTHETIC =', SYNTHETIC)

## 2. Statistik ringkas (median per fitur)

In [ ]:
med = pd.DataFrame({k: v[CANON].median() for k,v in domains.items()})
display(med.round(2))
RESULTS['median_per_feature'] = {k: {f: float(med.loc[f,k]) for f in CANON} for k in med.columns}

## 3. Jarak Wasserstein (W1) antar-domain (z-space)

In [ ]:
from scipy.stats import wasserstein_distance
from itertools import combinations
allX = pd.concat([v[CANON] for v in domains.values()], ignore_index=True)
mu, sd = allX.mean(), allX.std().replace(0,1)
Z = {k:(v[CANON]-mu)/sd for k,v in domains.items()}
pairs = list(combinations(domains.keys(),2))
rows=[]
for a,b in pairs:
    r={'pasangan':f'{a} vs {b}'}
    for c in CANON: r[c]=float(wasserstein_distance(Z[a][c].values, Z[b][c].values))
    r['RATA2']=float(np.mean([r[c] for c in CANON])); rows.append(r)
W = pd.DataFrame(rows).set_index('pasangan'); display(W.round(3))
RESULTS['wasserstein'] = W.reset_index().to_dict(orient='records')

fig,ax=plt.subplots(figsize=(7.2,3.4)); W['RATA2'].plot(kind='barh',ax=ax,color='#4C72B0')
ax.set_xlabel('W1 rata-rata (z-space)'); ax.set_title('Jarak distribusi antar-domain')
for i,(idx,val) in enumerate(W['RATA2'].items()): ax.text(val,i,f' {val:.2f}',va='center')
plt.tight_layout(); savefig('shift_wasserstein.png')

fig,ax=plt.subplots(figsize=(9,2.6+0.4*len(pairs))); M=W[CANON].values
im=ax.imshow(M,aspect='auto',cmap='YlOrRd'); ax.set_xticks(range(len(CANON))); ax.set_xticklabels(CANON,rotation=45,ha='right')
ax.set_yticks(range(len(W.index))); ax.set_yticklabels(W.index)
for i in range(M.shape[0]):
    for j in range(M.shape[1]): ax.text(j,i,f'{M[i,j]:.1f}',ha='center',va='center',fontsize=8)
fig.colorbar(im,ax=ax,label='W1'); ax.set_title('W1 per fitur x pasangan'); plt.tight_layout(); savefig('shift_wasserstein_heatmap.png')

## 4. ECDF fitur kunci

In [ ]:
def ecdf(x):
    x=np.sort(np.asarray(x,float)); return x, np.arange(1,len(x)+1)/len(x)
key=['duration','src_load','dst_load','fwd_pkts']
fig,axes=plt.subplots(2,2,figsize=(11,7))
for ax,feat in zip(axes.ravel(),key):
    for k,v in domains.items():
        val=v[feat].replace([np.inf,-np.inf],np.nan).dropna(); val=val[val>=0]
        if len(val)==0: continue
        xs,ys=ecdf(np.log1p(val)); ax.plot(xs,ys,label=k,color=colors.get(k),lw=2)
    ax.set_title(f'ECDF {feat} (log1p)'); ax.set_xlabel('log1p(nilai)'); ax.set_ylabel('proporsi<=x'); ax.legend()
plt.suptitle('ECDF fitur kunci: kurva terpisah = distribusi berbeda',fontsize=12); plt.tight_layout(); savefig('shift_ecdf.png')

## 5. Boxplot per fitur (log1p)

In [ ]:
fig,axes=plt.subplots(3,3,figsize=(12,9))
for ax,feat in zip(axes.ravel(),CANON):
    data=[]; labels=[]
    for k,v in domains.items():
        val=v[feat].replace([np.inf,-np.inf],np.nan).dropna(); data.append(np.log1p(val[val>=0].values)); labels.append(k)
    ax.boxplot(data,labels=labels,showfliers=False); ax.set_title(f'{feat} (log1p)')
plt.suptitle('Boxplot per fitur antar-domain',fontsize=12); plt.tight_layout(); savefig('shift_boxplot.png')

## 6. PCA & t-SNE 2D

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
CAP=1500; parts=[]; labs=[]; rng=np.random.default_rng(0)
for k,v in domains.items():
    d=v[CANON].replace([np.inf,-np.inf],np.nan).dropna()
    if len(d)>CAP: d=d.iloc[rng.choice(len(d),CAP,replace=False)]
    parts.append(d.values); labs+=[k]*len(d)
X=np.vstack(parts); labs=np.array(labs)
Xs=StandardScaler().fit_transform(np.log1p(np.clip(X,0,None)))
pca=PCA(n_components=2,random_state=0).fit_transform(Xs)
fig,ax=plt.subplots(figsize=(6.6,5))
for k in domains:
    m=labs==k; ax.scatter(pca[m,0],pca[m,1],s=8,alpha=0.4,label=k,color=colors.get(k))
ax.set_title('PCA 2D (9 fitur, log1p+z)'); ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.legend(); plt.tight_layout(); savefig('shift_pca.png')
try:
    from sklearn.manifold import TSNE
    ts=TSNE(n_components=2,perplexity=30,init='pca',random_state=0).fit_transform(Xs)
    fig,ax=plt.subplots(figsize=(6.6,5))
    for k in domains:
        m=labs==k; ax.scatter(ts[m,0],ts[m,1],s=8,alpha=0.4,label=k,color=colors.get(k))
    ax.set_title('t-SNE 2D (9 fitur)'); ax.legend(); plt.tight_layout(); savefig('shift_tsne.png')
except Exception as e:
    print('t-SNE dilewati:', e)

## 7. Domain classifier (proxy A-distance)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
def gap(a,b,n=1500):
    A=domains[a][CANON].replace([np.inf,-np.inf],np.nan).dropna().sample(min(n,len(domains[a])),random_state=0)
    B=domains[b][CANON].replace([np.inf,-np.inf],np.nan).dropna().sample(min(n,len(domains[b])),random_state=0)
    X=np.log1p(np.clip(np.vstack([A.values,B.values]),0,None)); y=np.r_[np.zeros(len(A)),np.ones(len(B))]
    clf=RandomForestClassifier(n_estimators=100,max_depth=8,random_state=0,n_jobs=-1)
    acc=float(cross_val_score(clf,X,y,cv=5,scoring='accuracy').mean())
    return acc, max(0.0, 2*(1-2*(1-acc)))
rows=[]
for a,b in pairs:
    acc,ad=gap(a,b); rows.append({'pasangan':f'{a} vs {b}','akurasi_pembeda':round(acc,3),'proxy_A_distance':round(ad,3)})
dg=pd.DataFrame(rows).set_index('pasangan'); display(dg)
RESULTS['domain_classifier'] = dg.reset_index().to_dict(orient='records')

## 8. Simpan artefak & UPLOAD ke S3

Menyimpan `dataset_shift_results.json` (semua angka) + PNG ke `shift_out/`, lalu unggah ke
`s3://<bucket>/unsw-far/shift/`. Artefak ini yang diunduh di PC lokal untuk analisis lanjutan.

In [ ]:
# Simpan JSON hasil
json_path = os.path.join(OUTDIR,'dataset_shift_results.json')
with open(json_path,'w') as f: json.dump(RESULTS, f, indent=2)
print('Tersimpan:', json_path)
print(json.dumps({k:RESULTS[k] for k in ['generated','synthetic_cic_uns','n_flow'] if k in RESULTS}, indent=2))

# Upload semua artefak (json + png) ke S3
try:
    import boto3
    s3 = boto3.client('s3', region_name=REGION)
    up = 0
    for fn in sorted(os.listdir(OUTDIR)):
        if fn.endswith(('.json','.png')):
            s3.upload_file(os.path.join(OUTDIR,fn), S3_BUCKET, f'{S3_PREFIX}/shift/{fn}')
            up += 1
    print(f'Upload {up} artefak -> s3://{S3_BUCKET}/{S3_PREFIX}/shift/')
    print('Isi folder shift/ di S3:')
    for o in s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=f'{S3_PREFIX}/shift/').get('Contents',[]):
        print('  ', o['Key'], o['Size'],'B')
except Exception as e:
    print('Upload S3 gagal (cek kredensial/izin):', e)
print('\nSELESAI. Beri tahu asisten -> unduh s3://%s/%s/shift/ untuk analisis lokal.' % (S3_BUCKET,S3_PREFIX))